# PathSnap - EDA, preprocessing and features (P1, report section 2.2)

Every table and plot states which partition it uses: **train** (80 trips) or **student_test** (10 trips).
The 10 evaluator trips are never used here. Outputs go to `results/eda/`.

In [1]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import display
from data_processing.load import DATA_ROOT, REPO_ROOT, list_trips, load_trip, LocalProjector
from features.candidates import TripGeometry

OUT = REPO_ROOT / "results" / "eda"; OUT.mkdir(parents=True, exist_ok=True)
PARTS = {"train": DATA_ROOT / "train", "student_test": DATA_ROOT / "student_test"}
print("data root:", DATA_ROOT)

data root: D:\College\sem 5\ML\Hackathon\Student folder\Student folder


## 1. Trips per partition and GPS points per trip

In [2]:
rows, trips_cache = [], {}
for part, d in PARTS.items():
    for p in list_trips(d):
        tr = load_trip(p)            # also validates columns, dtypes, timestamp order
        trips_cache[(part, tr["trip_id"])] = tr
        t, nd, ar = tr["track"], tr["nodes"], tr["arcs"]
        dt = np.diff(t["t"].to_numpy())
        rows.append(dict(partition=part, trip_id=tr["trip_id"], n_points=len(t), n_nodes=len(nd), n_arcs=len(ar),
                         n_route_arcs=len(tr["route"]), duration_s=t["t"].iloc[-1]-t["t"].iloc[0],
                         n_gaps_gt1s=int((dt > 1).sum()), max_gap_s=dt.max(),
                         lon_min=t.lon.min(), lon_max=t.lon.max(), lat_min=t.lat.min(), lat_max=t.lat.max(),
                         net_lon_min=nd.lon.min(), net_lon_max=nd.lon.max(), net_lat_min=nd.lat.min(), net_lat_max=nd.lat.max()))
trips = pd.DataFrame(rows)
print("load_trip() loaded", len(trips), "trips without errors")
display(trips.groupby("partition").agg(n_trips=("trip_id", "size"), points_total=("n_points", "sum"),
        points_mean=("n_points", "mean"), points_min=("n_points", "min"), points_max=("n_points", "max")).round(1))
fig, ax = plt.subplots(figsize=(8, 3.5))
for part, g in trips.groupby("partition"):
    ax.hist(g.n_points, bins=30, alpha=.6, label=f"{part} ({len(g)} trips)")
ax.set_xscale("log"); ax.set_xlabel("GPS points per trip (log)"); ax.set_ylabel("trips"); ax.legend()
ax.set_title("GPS points per trip - partitions: train, student_test"); fig.tight_layout(); fig.savefig(OUT / "points_per_trip.png", dpi=130); plt.show()
trips.to_csv(OUT / "trip_summary.csv", index=False)

load_trip() loaded 90 trips without errors


,n_trips,points_total,points_mean,points_min,points_max
partition,,,,,
student_test,10,19160,1916.0,262,3958
train,80,208417,2605.2,361,19313


## 2. Missing or invalid coordinates (partitions: train + student_test)

In [3]:
rows = []
for (part, tid), tr in trips_cache.items():
    t, nd = tr["track"], tr["nodes"]
    rows.append(dict(partition=part, trip_id=tid,
        track_nan=int(t.isna().sum().sum()), nodes_nan=int(nd.isna().sum().sum()),
        track_bad_range=int(((t.lon.abs() > 180) | (t.lat.abs() > 90)).sum()),
        nodes_bad_range=int(((nd.lon.abs() > 180) | (nd.lat.abs() > 90)).sum()),
        track_zero_zero=int(((t.lon == 0) & (t.lat == 0)).sum()),
        dup_timestamps=int(t.t.duplicated().sum()),
        arcs_self_loops=int((tr["arcs"].from_node == tr["arcs"].to_node).sum()),
        arcs_duplicates=int(tr["arcs"].duplicated().sum()),
        route_arcs_not_in_network=int((tr["route"] >= len(tr["arcs"])).sum())))
inv = pd.DataFrame(rows); inv.to_csv(OUT / "invalid_checks.csv", index=False)
print("Columns summed over all trips:"); display(inv.drop(columns=["partition", "trip_id"]).sum().to_frame("count").T)
print("Trips with any NaN / out-of-range coordinate:", int(((inv.track_nan + inv.nodes_nan + inv.track_bad_range + inv.nodes_bad_range) > 0).sum()))

Columns summed over all trips:


,track_nan,nodes_nan,track_bad_range,nodes_bad_range,track_zero_zero,dup_timestamps,arcs_self_loops,arcs_duplicates,route_arcs_not_in_network
count,0,0,0,0,0,0,24,1903,0


Trips with any NaN / out-of-range coordinate: 0


## 3. Time gaps (partitions: train + student_test)
The plan says GPS is ~1 Hz; gaps over 1 s are flagged.

In [4]:
gaps = {part: np.concatenate([np.diff(tr["track"].t.to_numpy()) for (p, _), tr in trips_cache.items() if p == part]) for part in PARTS}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (part, g) in zip(axs, gaps.items()):
    vals, cnt = np.unique(np.clip(g, 0, 20), return_counts=True)
    ax.bar(vals, cnt, width=.8); ax.set_yscale("log"); ax.axvline(1.5, color="r", ls="--", label="gap > 1 s")
    ax.set_title(f"Time gap between fixes - {part}"); ax.set_xlabel("seconds (clipped at 20)"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "time_gap_hist.png", dpi=130); plt.show()
for part, g in gaps.items():
    print(f"{part}: {len(g)} gaps | exactly 1 s: {(g == 1).mean():.1%} | > 1 s: {(g > 1).mean():.2%} | max {g.max():.0f} s | non-positive: {(g <= 0).sum()}")
display(trips.sort_values("max_gap_s", ascending=False)[["partition", "trip_id", "n_points", "n_gaps_gt1s", "max_gap_s"]].head(8))

train: 208337 gaps | exactly 1 s: 96.5% | > 1 s: 3.50% | max 153064 s | non-positive: 0
student_test: 19150 gaps | exactly 1 s: 98.6% | > 1 s: 1.38% | max 210 s | non-positive: 0


,partition,trip_id,n_points,n_gaps_gt1s,max_gap_s
72,train,00000090,536,224,153064.0
61,train,00000078,2909,712,52810.0
55,train,00000069,2067,22,2658.0
16,train,00000021,15783,6,2161.0
8,train,00000011,2828,1,909.0
34,train,00000042,7739,81,670.0
64,train,00000081,3224,903,536.0
18,train,00000024,801,23,458.0


## 4. Bounding boxes, node and arc counts per trip

In [5]:
trips["track_span_km_lon"] = (trips.lon_max - trips.lon_min) * 111.32 * np.cos(np.radians((trips.lat_min + trips.lat_max) / 2))
trips["track_span_km_lat"] = (trips.lat_max - trips.lat_min) * 110.57
display(trips.groupby("partition")[["n_nodes", "n_arcs", "n_route_arcs", "track_span_km_lon", "track_span_km_lat"]].describe().T.round(1))
print("Track centres (lon range, lat range) - the dataset is worldwide, not only Germany:")
print(f"  lon {trips.lon_min.min():.1f} .. {trips.lon_max.max():.1f}   lat {trips.lat_min.min():.1f} .. {trips.lat_max.max():.1f}")
print("-> a single UTM zone is invalid for most trips; we use a per-trip local metric projection (LocalProjector).")
display(trips[["partition", "trip_id", "n_points", "n_nodes", "n_arcs", "n_route_arcs", "lon_min", "lon_max", "lat_min", "lat_max"]].head(10))

partition                student_test     train
n_nodes           count          10.0      80.0
                  mean         9068.9   24168.6
                  std          7452.2   32272.3
                  min          1770.0     299.0
                  25%          4908.0    3487.8
                  50%          6596.5    9608.0
                  75%         11844.2   28846.8
                  max         26635.0  155003.0
n_arcs            count          10.0      80.0
                  mean        18504.1   48461.3
                  std         15496.9   64738.8
                  min          3577.0     649.0
                  25%         10292.2    7061.8
                  50%         13145.0   19526.5
                  75%         24818.5   58003.2
                  max         56027.0  307392.0
n_route_arcs      count          10.0      80.0
                  mean          347.7     391.7
                  std           359.6     361.2
                  min            95.0      59.0
                  25%           153.5     168.2
                  50%           250.0     280.0
                  75%           357.8     480.5
                  max          1323.0    2266.0
track_span_km_lon count          10.0      80.0
                  mean            9.9      14.0
                  std            12.7      16.1
                  min             2.3       0.9
                  25%             3.2       3.3
                  50%             5.2       7.7
                  75%            10.6      16.6
                  max            44.3      70.4
track_span_km_lat count          10.0      80.0
                  mean            9.2      11.9
                  std             8.1      14.7
                  min             1.9       0.9
                  25%             3.7       3.1
                  50%             5.9       5.2
                  75%            11.4      13.9
                  max            26.7      75.0

Track centres (lon range, lat range) - the dataset is worldwide, not only Germany:
  lon -100.4 .. 137.2   lat -34.6 .. 61.7
-> a single UTM zone is invalid for most trips; we use a per-trip local metric projection (LocalProjector).


,partition,trip_id,n_points,n_nodes,n_arcs,n_route_arcs,lon_min,lon_max,lat_min,lat_max
0,train,00000001,1315,88809,165790,525,8.570524,8.647860,49.844053,50.211882
1,train,00000002,7243,7936,16139,262,10.999961,11.050902,47.658344,47.695459
2,train,00000003,821,9138,18028,79,37.154202,37.201595,55.977135,56.007585
3,train,00000004,1064,7350,15499,320,12.342280,12.459615,52.590308,52.718960
4,train,00000005,1045,898,1579,105,35.485577,35.544315,49.357581,49.613085
5,train,00000006,1450,3259,6757,261,37.276002,37.317675,55.429164,55.487229
6,train,00000008,3093,4995,10421,173,-86.584860,-86.567487,34.733275,34.753704
7,train,00000010,2711,8502,15715,210,37.357893,37.394790,55.786261,55.832857
8,train,00000011,2828,2649,5393,565,59.647057,60.016866,56.041788,56.067535
9,train,00000012,1608,16376,38599,205,44.044515,44.389906,36.175914,36.263587


## 5. Connected components and isolated nodes (networkx; partitions: train + student_test)

In [6]:
rows = []
for (part, tid), tr in trips_cache.items():
    G = nx.DiGraph(); G.add_nodes_from(range(len(tr["nodes"])))
    G.add_edges_from(zip(tr["arcs"].from_node.to_numpy(), tr["arcs"].to_node.to_numpy()))
    comps = [len(c) for c in nx.weakly_connected_components(G)]
    rows.append(dict(partition=part, trip_id=tid, n_nodes=G.number_of_nodes(), n_arcs=G.number_of_edges(),
        weak_components=len(comps), largest_component_share=max(comps) / G.number_of_nodes(),
        isolated_nodes=int(sum(1 for n in G if G.degree(n) == 0)),
        strong_components=nx.number_strongly_connected_components(G)))
cc = pd.DataFrame(rows); cc.to_csv(OUT / "connectivity.csv", index=False)
display(cc.groupby("partition")[["weak_components", "largest_component_share", "isolated_nodes", "strong_components"]].describe().T.round(3))
display(cc.sort_values("weak_components", ascending=False).head(6))

partition                      student_test     train
weak_components         count        10.000    80.000
                        mean         22.700    39.012
                        std          29.182    60.682
                        min           5.000     1.000
                        25%           7.250     9.750
                        50%          11.000    20.000
                        75%          21.750    42.250
                        max         100.000   421.000
largest_component_share count        10.000    80.000
                        mean          0.964     0.969
                        std           0.034     0.039
                        min           0.874     0.693
                        25%           0.964     0.963
                        50%           0.971     0.981
                        75%           0.983     0.987
                        max           0.991     1.000
isolated_nodes          count        10.000    80.000
                        mean          0.000     0.000
                        std           0.000     0.000
                        min           0.000     0.000
                        25%           0.000     0.000
                        50%           0.000     0.000
                        75%           0.000     0.000
                        max           0.000     0.000
strong_components       count        10.000    80.000
                        mean         97.400   250.475
                        std         125.957   343.237
                        min           6.000     2.000
                        25%          21.750    15.500
                        50%          53.500   127.000
                        75%          94.750   328.500
                        max         415.000  1942.000

,partition,trip_id,n_nodes,n_arcs,weak_components,largest_component_share,isolated_nodes,strong_components
35,train,00000043,109070,215298,421,0.977400,0,1058
44,train,00000054,63822,116820,213,0.970371,0,1182
69,train,00000086,114868,237593,193,0.981170,0,339
0,train,00000001,88809,165498,175,0.977986,0,1942
25,train,00000032,124540,259357,126,0.991111,0,732
28,train,00000035,21126,35174,119,0.914229,0,979


## 6. Distance from each GPS point to its reference route arc
Uses the labels built by `data_processing/labels.py` (forward-only alignment). Partitions: train and student_test.
Points whose reference arc is > 100 m away are flagged `label_valid = 0`: the `.route` does not cover them.

In [7]:
pl = {"train": pd.read_parquet(REPO_ROOT / "results" / "point_labels_train.parquet"),
      "student_test": pd.read_parquet(REPO_ROOT / "results" / "point_labels_studtest.parquet")}
fig, axs = plt.subplots(1, 2, figsize=(11, 3.6))
for ax, (part, p) in zip(axs, pl.items()):
    v = p[p.label_valid == 1].label_dist
    ax.hist(v, bins=60, range=(0, 60)); ax.axvline(v.median(), color="r", ls="--", label=f"median {v.median():.1f} m")
    ax.set_title(f"GPS point -> reference arc (valid points) - {part}"); ax.set_xlabel("metres"); ax.legend()
fig.tight_layout(); fig.savefig(OUT / "dist_to_route.png", dpi=130); plt.show()
rows = []
for part, p in pl.items():
    v = p[p.label_valid == 1].label_dist
    rows.append(dict(partition=part, points=len(p), label_valid=int(p.label_valid.sum()), invalid_share=1 - p.label_valid.mean(),
                     median_m=v.median(), p90_m=v.quantile(.9), p99_m=v.quantile(.99)))
display(pd.DataFrame(rows).round(3))
bad = pl["train"].groupby("trip_id").label_valid.mean(); print("train trips where the route covers < 90% of the track:", bad[bad < .9].round(2).to_dict())

,partition,points,label_valid,invalid_share,median_m,p90_m,p99_m
0,train,208417,187865,0.099,5.005,13.934,56.727
1,student_test,19160,19143,0.001,5.262,13.418,27.472


train trips where the route covers < 90% of the track: {'00000033': 0.49, '00000054': 0.15}


## 7. Candidate miss rate for several radii R (partition-labelled)
Miss rate = share of label-valid points whose reference arc is not among the candidates (nearest k=8 arcs within R).

In [8]:
mr = pd.read_csv(REPO_ROOT / "results" / "candidate_miss_rate.csv")
display(mr[mr.k.astype(str) == "8"].pivot(index="radius_m", columns="partition", values="miss_rate").round(4))
print("With no cap on k:"); display(mr[mr.k.astype(str) == "all"].pivot(index="radius_m", columns="partition", values="miss_rate").round(4))

partition,student_test,train
radius_m,,
10,0.2052,0.2005
20,0.0292,0.0539
30,0.0089,0.0335
50,0.0031,0.0228
75,0.0028,0.0177
100,0.0026,0.0141


With no cap on k:


partition,student_test,train
radius_m,,
10,0.2049,0.1994
20,0.0281,0.0509
30,0.0068,0.0290
50,0.0005,0.0139
75,0.0002,0.0045
100,0.0000,0.0000


## 8. Folium map - one train trip (GPS points, road network, reference route)
Layers: GPS points (blue), road network (grey, clipped to the track's surroundings), reference route (green). Legend + layer control.

In [9]:
import folium
from folium.plugins import MarkerCluster
tid = "00000052"; part = "student_test"      # a small trip for a light map; train/ has 80 more
tr = trips_cache[(part, tid)]
t, nd, ar = tr["track"], tr["nodes"], tr["arcs"]
m = folium.Map(location=[t.lat.mean(), t.lon.mean()], zoom_start=14, tiles="cartodbpositron")
box = (t.lon.min() - .004, t.lon.max() + .004, t.lat.min() - .004, t.lat.max() + .004)
near = (nd.lon.between(box[0], box[1]) & nd.lat.between(box[2], box[3])).to_numpy()
sel = ar[near[ar.from_node.to_numpy()] & near[ar.to_node.to_numpy()]]
net = folium.FeatureGroup(name=f"Road network ({len(sel)} arcs in view)", show=True)
for f, tt in zip(sel.from_node.to_numpy(), sel.to_node.to_numpy()):
    folium.PolyLine([(nd.lat[f], nd.lon[f]), (nd.lat[tt], nd.lon[tt])], color="#888888", weight=2, opacity=.7).add_to(net)
net.add_to(m)
rt = folium.FeatureGroup(name=f"Reference route ({len(tr['route'])} arcs)")
for a in tr["route"]:
    f, tt = ar.from_node[a], ar.to_node[a]
    folium.PolyLine([(nd.lat[f], nd.lon[f]), (nd.lat[tt], nd.lon[tt])], color="green", weight=5, opacity=.9).add_to(rt)
rt.add_to(m)
gp = folium.FeatureGroup(name=f"GPS points ({len(t)})")
for la, lo in zip(t.lat, t.lon):
    folium.CircleMarker((la, lo), radius=2, color="blue", fill=True, fill_opacity=.8, weight=1).add_to(gp)
gp.add_to(m)
legend = '''<div style="position:fixed;bottom:30px;left:30px;z-index:9999;background:white;padding:8px 12px;border:1px solid #999;font-size:13px">
<b>Legend</b><br><span style="color:blue">&#9679;</span> GPS points<br><span style="color:#888">&#9473;</span> Road network<br><span style="color:green">&#9473;</span> Reference route</div>'''
m.get_root().html.add_child(folium.Element(legend))
folium.LayerControl(collapsed=False).add_to(m)
(REPO_ROOT / "results" / "maps").mkdir(exist_ok=True)
m.save(REPO_ROOT / "results" / "maps" / f"eda_{part}_{tid}.html"); print("saved eda map for", part, tid)

saved eda map for student_test 00000052


## 9. Preprocessing summary and feature notes
* **Projection**: per-trip local azimuthal-equidistant (metres). Distances are never measured in degrees.
* **Labels**: forward-only reference arcs (`data_processing/labels.py`), `label_valid` flags points the route does not cover.
* **Candidates**: arcs within R=50 m, at most k=8 nearest, via a shapely STRtree (`features/candidates.py`).
* **Features**: see the module docstring of `features/build_features.py` (also `results/feature_notes.md`).